# Student Exercises & Extensions (Bootcamp-Ready)

**Purpose:** Turn the course into something students can *do*, not just watch.

This notebook is organized by difficulty:
- **Core (Undergrad-friendly)**
- **Intermediate (Working/Grad)**
- **Advanced (PhD/Research)**

Each section includes:
- student checklist
- templates for recording results
- suggested deliverables

---

## What you should already have by now
From the earlier notebooks you should understand:
- embeddings + similarity metrics
- FAISS indexing and semantic search
- failure cases and evaluation basics
- metadata filtering
- mini-RAG (retrieve → answer grounded)

Now you will practice building and measuring.


## 0) Setup

You can reuse code patterns from previous notebooks.
This notebook provides a lightweight scaffold so you can run everything in one place.

If you're using the earlier notebooks as modules, you can copy/paste their helper functions here.


In [ ]:
# Install dependencies if needed (safe to re-run)
try:
    import sentence_transformers  # noqa: F401
except ImportError:
    !uv add sentence-transformers

try:
    import faiss  # noqa: F401
except ImportError:
    !uv add install faiss-cpu

In [ ]:
import time
import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer

pd.set_option("display.max_colwidth", 150)
np.random.seed(42)

## 1) Starter corpus (you will extend this)

You will:
- add **50 new documents**
- write **10 evaluation queries**
- measure **Precision@5**
- compare **two embedding models**

You can replace this corpus with your own dataset.


In [ ]:
docs = [
    ("finance", "Reduce spending by reviewing subscriptions and recurring bills."),
    ("finance", "Banks assess credit risk before approving loans."),
    ("finance", "Pay down high-interest debt first to save on interest payments."),
    ("software", "Refactor code to reduce technical debt and improve maintainability."),
    ("software", "Profile your program to find performance bottlenecks."),
    ("software", "Add caching to avoid recomputing expensive results."),
    ("ml_ai", "Embeddings represent text as vectors so similar meanings are close."),
    ("ml_ai", "Overfitting happens when a model memorizes training data."),
    ("weather", "Monitor official advisories when a typhoon is nearby."),
    ("weather", "Avoid driving through flooded roads during heavy rain."),
    ("business", "A loyalty program can increase repeat purchases."),
    ("business", "Customer retention improves when support resolves issues quickly."),
    ("health", "Breathing exercises can reduce stress in the short term."),
    ("health", "Prioritize sleep to support focus and memory."),
]

df = pd.DataFrame(docs, columns=["topic", "text"])
df.insert(0, "doc_id", [f"D{i:04d}" for i in range(len(df))])
df

## Helper functions (retrieval + evaluation)

We use:
- normalized embeddings
- FAISS IndexFlatIP (cosine-like)

We will compute:
- Precision@5 for each query
- Mean Precision@5 across queries

Labeling approach:
- For each query, you manually provide a set of relevant `doc_id`s.


In [ ]:
def build_index(model_name: str, texts: list[str]):
    model = SentenceTransformer(model_name)
    emb = model.encode(texts, normalize_embeddings=True).astype("float32")
    dim = emb.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(emb)
    return model, index, emb

def retrieve_topk(model, index, query: str, k=5):
    q = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, idx = index.search(q, k)
    return scores[0], idx[0]

def precision_at_k(retrieved_doc_ids, relevant_doc_ids, k=5):
    if k == 0:
        return 0.0
    hits = sum(1 for d in retrieved_doc_ids[:k] if d in relevant_doc_ids)
    return hits / k

def evaluate_precision_at_k(df, model_name, labeled_queries, k=5):
    texts = df["text"].tolist()
    model, index, _ = build_index(model_name, texts)
    
    rows = []
    for item in labeled_queries:
        qid = item["query_id"]
        query = item["query"]
        relevant = set(item["relevant_doc_ids"])
        
        t0 = time.time()
        scores, idx = retrieve_topk(model, index, query, k=k)
        latency_ms = (time.time() - t0) * 1000.0
        
        retrieved_doc_ids = [df.loc[i, "doc_id"] for i in idx]
        p_at_k = precision_at_k(retrieved_doc_ids, relevant, k=k)
        
        rows.append({
            "query_id": qid,
            "query": query,
            f"Precision@{k}": p_at_k,
            "latency_ms": latency_ms,
            "retrieved_doc_ids": retrieved_doc_ids,
            "relevant_doc_ids": sorted(list(relevant)),
        })
        
    results = pd.DataFrame(rows)
    summary = pd.DataFrame([{
        "model": model_name,
        f"MeanPrecision@{k}": float(results[f"Precision@{k}"].mean()),
        "AvgLatency(ms)": float(results["latency_ms"].mean())
    }])
    return results, summary

# CORE EXERCISES (Undergrad-Friendly)

## ✅ Exercise 1: Add 50 new documents to the corpus

**Checklist**
- [ ] Add at least **50 new short texts** (1–2 sentences each)
- [ ] Keep them diverse across 3+ topics
- [ ] Ensure some paraphrases / synonyms exist (so semantic search has something to do)

**Tip:** Copy patterns from your notes:
- software performance
- finance budgeting
- weather preparedness
- customer retention
- health habits


In [ ]:
# TODO: Add 50 new docs here
# Format: ("topic", "text")
# Example:
# new_docs = [
#     ("finance", "Automate savings transfers so you don't forget."),
#     ("software", "Use database indexes to speed up lookups."),
#     ...
# ]
new_docs = []  # <-- add at least 50

# Append and rebuild df
if len(new_docs) > 0:
    df2 = pd.DataFrame(new_docs, columns=["topic", "text"])
    df = pd.concat([df, df2], ignore_index=True)
    df["doc_id"] = [f"D{i:04d}" for i in range(len(df))]

print("Corpus size:", len(df))
df.tail()

## ✅ Exercise 2: Write 10 queries and evaluate Precision@5

You will create **10 queries** and label relevant documents.

**Checklist**
- [ ] Write **10 queries** (some short, some long)
- [ ] For each query, list **relevant_doc_ids**
- [ ] Run evaluation and compute **Mean Precision@5**

**Labeling rules (simple)**
- A retrieved doc is relevant if it would help answer the query.
- It’s okay if multiple docs are relevant.


In [ ]:
# TODO: Fill in 10 queries and relevant_doc_ids
# Use doc_ids from df (e.g., "D0007")
#
# Tip: Run df[df['topic']=='finance'].head() to see doc_ids after you add docs.

labeled_queries = [
    # Example template:
    # {"query_id": "Q01", "query": "How do I reduce spending fast?", "relevant_doc_ids": ["D0000", "D0002"]},
]

# Safety check
print("Labeled queries:", len(labeled_queries))
if len(labeled_queries) > 0:
    pd.DataFrame(labeled_queries)

In [ ]:
# Evaluate Precision@5 using one baseline model (once you filled labeled_queries)
BASE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

if len(labeled_queries) == 0:
    print("Fill labeled_queries first.")
else:
    per_query, summary = evaluate_precision_at_k(df, BASE_MODEL, labeled_queries, k=5)
    display(summary)
    display(per_query[["query_id", "Precision@5", "latency_ms", "retrieved_doc_ids", "relevant_doc_ids"]])

## ✅ Exercise 3: Try a second embedding model and compare

Compare:
- Model A: `all-MiniLM-L6-v2` (fast)
- Model B: `all-mpnet-base-v2` (often stronger)

**Checklist**
- [ ] Evaluate both models on the same labeled queries
- [ ] Compare Mean Precision@5 and latency
- [ ] Identify which queries improved and which got worse


In [ ]:
MODEL_A = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_B = "sentence-transformers/all-mpnet-base-v2"

if len(labeled_queries) == 0:
    print("Fill labeled_queries first.")
else:
    res_a, sum_a = evaluate_precision_at_k(df, MODEL_A, labeled_queries, k=5)
    res_b, sum_b = evaluate_precision_at_k(df, MODEL_B, labeled_queries, k=5)
    
    metrics = pd.concat([sum_a, sum_b], ignore_index=True)
    display(metrics)
    
    # Before vs after by query
    compare = res_a[["query_id", "query", "Precision@5"]].merge(
        res_b[["query_id", "Precision@5"]],
        on="query_id",
        suffixes=("_A", "_B")
    )
    compare["delta(B-A)"] = compare["Precision@5_B"] - compare["Precision@5_A"]
    display(compare.sort_values("delta(B-A)", ascending=False))

# INTERMEDIATE (Working / Grad)

## ✅ Exercise 4: Add metadata and implement filtering

**Goal:** Show that retrieval isn’t only vectors.

**Checklist**
- [ ] Add metadata fields to your corpus (at least 2 of these):
  - `date` (or recency bucket)
  - `source` (handbook/wiki/support/news)
  - `category` (already have `topic`)
- [ ] Add a filtered search mode:
  - filter by metadata first (or post-filter)
  - then vector search
- [ ] Evaluate Precision@5 on scoped queries (e.g., “from handbook”, “latest”, “finance only”)

**Deliverable**
- A short table showing **with vs without filters**
- A sentence explaining the precision/recall tradeoff


In [ ]:
# TODO: Add metadata columns
# Example scaffolding:
#
# df["source"] = np.random.choice(["handbook", "wiki", "support"], size=len(df))
# df["date"] = ... (e.g., random recent dates)
#
# Then implement:
# - unfiltered retrieval
# - filtered retrieval (pre/post filter)
#
# Keep it simple: filter to subset rows, then build a temporary index on the subset.
pass

## ✅ Exercise 5: Add a reranker idea (concept-only OR optional implementation)

A common improvement pattern:
1. Retrieve top-50 by embeddings (fast, high recall)
2. **Rerank** those 50 using a smarter method (slower, but only on small set)

Reranker options:
- **Cross-encoder** model (best classic approach)
- **LLM reranker** (careful; can be expensive and inconsistent)
- **Heuristic reranker** (metadata boosts: recency, trusted source, etc.)

**Checklist**
- [ ] Write a short paragraph explaining your reranker choice
- [ ] Define what it optimizes (precision? intent match? recency?)
- [ ] (Optional) Implement a simple heuristic reranker:
  - boost score if source == handbook
  - boost score if date within last 30 days
  - etc.

**Deliverable**
- Before vs after top-5 results for 3 queries
- Short explanation of why reranking changed results


In [ ]:
# OPTIONAL: Simple heuristic reranker scaffold (no LLM)
# Steps:
# 1) Retrieve top_n candidates by embeddings
# 2) Apply adjusted_score = score + boosts
# 3) Sort by adjusted_score, return top_k
#
# You can use metadata boosts like:
# - +0.05 if source == 'handbook'
# - +0.05 if date is recent
pass

# ADVANCED (PhD / Research)

## ✅ Exercise 6: Design an experiment

This is the “research mindset” module.

### Step 1 — Define objective
Examples:
- Improve Precision@5 for finance queries
- Reduce latency while keeping Precision@5 within 95% of baseline
- Improve performance on slang queries

### Step 2 — Define metric
Choose 1–2:
- Precision@k, Recall@k, MRR
- latency
- slice performance (e.g., slang-only queries)

### Step 3 — Compare two variants
Examples:
- Encoder A vs Encoder B
- With metadata filter vs no filter
- With reranker vs no reranker
- Different chunking strategies (if you have real docs)

### Step 4 — Write findings paragraph
A short “paper-style” paragraph:
- what you compared
- what metric changed
- what tradeoff appeared
- what you recommend next


## Experiment template (copy/paste)

Fill this out and submit as your result.

### Experiment Title
**(e.g., “Effect of metadata filtering on finance query precision”)**

### Objective
- (one sentence)

### Dataset
- corpus size:
- query set size:
- labeling method:

### Variants
- Variant A:
- Variant B:

### Metrics
- Primary metric:
- Secondary metric:

### Results (numbers)
- A:
- B:
- Delta:

### Findings paragraph (5–8 sentences)
Write a short, concrete summary of what happened and why.


In [ ]:
# Optional: Results worksheet table you can fill programmatically
results_template = pd.DataFrame([
    {"field": "Experiment Title", "value": ""},
    {"field": "Objective", "value": ""},
    {"field": "Corpus Size", "value": ""},
    {"field": "Query Set Size", "value": ""},
    {"field": "Labeling Method", "value": ""},
    {"field": "Variant A", "value": ""},
    {"field": "Variant B", "value": ""},
    {"field": "Primary Metric", "value": ""},
    {"field": "Secondary Metric", "value": ""},
    {"field": "Result A", "value": ""},
    {"field": "Result B", "value": ""},
    {"field": "Delta (B-A)", "value": ""},
    {"field": "Findings Paragraph", "value": ""},
])
results_template

## Outputs checklist

- ✅ student checklists (core/intermediate/advanced)
- ✅ templates for writing results (experiment template + findings paragraph)
- ✅ scaffolding code for evaluation + comparisons
